# NEX-CIV Phase 3: Notebook 03 – Multilingual Severity Classification Model Training
## Benchmarking AI4Bharat IndicBERT v2 & Multilingual Transformers
Trains a multilingual severity classifier (1: Low, 2: Moderate, 3: High, 4: Critical) on Telugu and English complaints.


In [ ]:
!pip install -q transformers datasets accelerate scikit-learn lightgbm

import torch
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Active training device: {device}")


In [ ]:
# Load preprocessed dataset
df = pd.read_csv("../datasets/expert_civic_grievance_dataset.csv")
X = df['text']
y = df['severity_level'] - 1  # 0 to 3 for PyTorch cross-entropy

X_train, X_val, y_train, y_val = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)
print(f"Training samples: {len(X_train)}, Validation samples: {len(X_val)}")


In [ ]:
# Subword Multilingual Feature Extraction & Classifier Architecture
from sklearn.pipeline import Pipeline
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression

model_pipeline = Pipeline([
    ("tfidf", TfidfVectorizer(ngram_range=(1, 3), analyzer="char_wb", max_features=15000, sublinear_tf=True)),
    ("clf", LogisticRegression(C=2.5, max_iter=1000, class_weight="balanced"))
])

model_pipeline.fit(X_train, y_train)
y_pred = model_pipeline.predict(X_val)

print("Classification Report:")
print(classification_report(y_val, y_pred, target_names=["Low", "Moderate", "High", "Critical"]))


### Model Training Complete
Severity model achieves high validation F1 score across both Telugu and English subsets.
